<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks/aula_17_clustering_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Setting
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import numpy as np
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter

In [3]:
# Data import
url_roll_calls = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-03-23/roll_calls.csv"
df_roll = pd.read_csv(url_roll_calls)

In [4]:
# Data validate
print(df_roll.shape)
print(f"Number of resolutioins: {df_roll['rcid'].nunique()}")
print(f"NA rows: {df_roll['descr'].isna().sum()}")
df_roll.head(2)

(6202, 9)
Number of resolutioins: 6202
NA rows: 1


,rcid,session,importantvote,date,unres,amend,para,short,descr
0,3,1,0.0,1946-01-01,R/1/66,1.0,0.0,"AMENDMENTS, RULES OF PROCEDURE",TO ADOPT A CUBAN AMENDMENT TO THE UK PROPOSAL ...
1,4,1,0.0,1946-01-02,R/1/79,0.0,0.0,SECURITY COUNCIL ELECTIONS,TO ADOPT A USSR PROPOSAL ADJOURNING DEBATE ON ...


In [5]:
# Data Cleaning
df_roll_clean = df_roll[~df_roll['descr'].isna()]

In [6]:
# Data features/treatment

# Time data format
df_roll_clean['year'] = pd.to_datetime(df_roll_clean['date']).dt.year

# Remover números
df_roll_clean['descr_clean'] = df_roll_clean['descr'].str.replace(r'\d+', '', regex=True)

# Converter para minúsculo
df_roll_clean['descr_clean'] = df_roll_clean['descr_clean'].str.lower()

# Remover quebras de linha e barras invertidas
df_roll_clean['descr_clean'] = df_roll_clean['descr_clean'].str.replace('\\', ' ', regex=False).str.replace('\n', ' ', regex=False)

# Remover caracteres especiais e pontuação
df_roll_clean['descr_clean'] = df_roll_clean['descr_clean'].str.replace(r'[^a-zA-Z\s]', '', regex=True)

# Remover excesso de espaços em branco
df_roll_clean['descr_clean'] = df_roll_clean['descr_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Mostrar o resultado final
df_roll_clean['descr_clean'].head()

# Definir conjunto de stopwords padrão em inglês e adicionar palavras personalizadas
custom_stopwords = set(ENGLISH_STOP_WORDS)
custom_stopwords.update(['adopt', 'adopted', 'adopting',
                         'proposal', 'amendment', 'resolution'])
df_roll_clean['word'] = df_roll_clean['descr_clean'].str.split()

df_roll_clean['descr_clean'] = [
    " ".join([word for word in text if word not in custom_stopwords])
    for text in df_roll_clean['word']
]

# Term Frequency
tf_vectorizer = CountVectorizer(stop_words='english', max_df=0.95, min_df=2)
X_tf = tf_vectorizer.fit_transform(df_roll_clean['descr_clean'])
terms = tf_vectorizer.get_feature_names_out()
df_tf = pd.DataFrame(X_tf.toarray(), columns=terms)
df_tf.head(3)

/tmp/ipykernel_18961/3327257925.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_roll_clean['year'] = pd.to_datetime(df_roll_clean['date']).dt.year
/tmp/ipykernel_18961/3327257925.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_roll_clean['descr_clean'] = df_roll_clean['descr'].str.replace(r'\d+', '', regex=True)
/tmp/ipykernel_18961/3327257925.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

,aa,aadd,ababa,abandon,abide,abkhazia,able,abolish,abolition,abroad,...,yrs,yugoslavia,zambia,zealand,zimbabwe,zimbabwese,zionism,zone,zones,zonesouthern
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# Modelagem
best_eps = 1
best_min_samples = 10
db_final = DBSCAN(eps=best_eps, min_samples=best_min_samples)
labels = db_final.fit_predict(df_tf)
df_roll_clean['cluster'] = labels
score = silhouette_score(df_tf, labels)

/tmp/ipykernel_18961/1291519680.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_roll_clean['cluster'] = labels


In [10]:
# Validação
labels_lista = df_roll_clean['cluster'].unique()
n_clusters_ = len([x for x in labels_lista if x != -1])
n_noise_ = list(df_tf['cluster']).count(-1)

print(f"Número total de clusters encontrados (excluindo ruído): {n_clusters_}")
print(f"Número de pontos classificados como ruído (-1): {n_noise_}")
print(f"Silhouette score: {score:.4f}")

Número total de clusters encontrados (excluindo ruído): 80
Número de pontos classificados como ruído (-1): 0
Silhouette score: -0.1353


In [11]:
# Calcular a frequência absoluta e relativa dos clusters
freqa_abs = df_roll_clean['cluster'].value_counts(dropna=False)
freq_rel = df_roll_clean['cluster'].value_counts(normalize=True, dropna=False) * 100

# Criar um DataFrame de resumo e ordená-lo de forma decrescente pela frequência
resumo_clusters = pd.DataFrame({
    'Frequência Absoluta': freqa_abs,
    'Frequência Relativa (%)': freq_rel
}).sort_values(by='Frequência Absoluta', ascending=False)

resumo_clusters.index.name = 'Cluster'
print("Distribuição e Frequência dos Clusters (Ordenados do maior para o menor):")
display(resumo_clusters)

Distribuição e Frequência dos Clusters (Ordenados do maior para o menor):


,Frequência Absoluta,Frequência Relativa (%)
Cluster,,
-1,5126,82.664086
1,31,0.499919
79,25,0.403161
47,25,0.403161
6,24,0.387034
...,...,...
50,10,0.161264
56,10,0.161264
63,10,0.161264


In [ ]:
# Definir os subplots (uma linha com 5 colunas ou conforme apropriado)
fig, axes = plt.subplots(1, n_clusters_+1, figsize=(25, 5))
fig.suptitle('Nuvens de Palavras: 10 Termos mais Frequentes por Cluster', fontsize=16, fontweight='bold', y=1.05)

for cluster_num in range(-1,n_clusters_):
    print(f"Cluster {cluster_num}:")
    # Filtrar os textos pertencentes ao cluster atual
    textos_cluster = df_roll_clean[df_roll_clean['cluster'] == cluster_num]['descr_clean']

    # Unir e tokenizar as palavras para contar a frequência
    todas_palavras = " ".join(textos_cluster).split()

    # Pegar as 10 palavras mais frequentes
    contagem = Counter(todas_palavras)
    top_10 = dict(contagem.most_common(10))

    # Criar a nuvem de palavras baseada na frequência dos top 10 termos
    wordcloud = WordCloud(
        width=400,
        height=400,
        background_color='white',
        colormap='viridis'
    ).generate_from_frequencies(top_10)

    # Plotar o gráfico no respectivo subplot
    ax = axes[cluster_num]
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.set_title(f'Cluster {cluster_num}', fontsize=14, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()